# Counterfactual Inference Tutorial

This notebook shows how to run counterfactual load forecasting with a trained NACF checkpoint. It does not train the model.

It demonstrates three inference settings for the same historical load/weather/calendar context:

- factual prediction with the observed news events,
- no-news counterfactual prediction,
- custom-news scenario prediction, where you write the hypothetical news events yourself.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from types import SimpleNamespace

from dataset import get_dataloaders
from model import NACFModel
from utils import inverse_transform_predictions, extract_sample_news

## Configuration

The released repository includes a public inference checkpoint. If you want to use a different local checkpoint, set `checkpoint_path` manually.


In [ ]:
checkpoint_path = PROJECT_DIR / "weights" / "nacf_nsw_2019_public.pth"
if not checkpoint_path.exists():
    raise FileNotFoundError(
        "Public checkpoint not found at weights/nacf_nsw_2019_public.pth."
    )


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

args = SimpleNamespace(
    state="NSW",
    data_path="./data/nsw_2019_structured_events.csv",
    train_ratio=0.7,
    val_ratio=0.1,
    test_ratio=0.2,
    seq_len=48,
    pred_len=48,
    sampling_stride=1,
    enc_in=10,
    d_model=512,
    n_heads=8,
    e_layers=2,
    d_ff=2048,
    dropout=0.1,
    activation="gelu",
    treat_hidden=128,
    lambda_ipm=1.2,
    rbf_sigma=8.0,
    num_bins=20,
    batch_size=1,
    num_workers=0,
    device=device,
)

device


## Load Data And Model

In [ ]:
import os
os.chdir(PROJECT_DIR)

_, _, test_loader, scaler = get_dataloaders(args)

model = NACFModel(args).to(device)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
missing_keys = list(load_result.missing_keys)
unexpected_keys = list(load_result.unexpected_keys)
if unexpected_keys:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_keys}")
if missing_keys and not all(key.startswith("news_encoder.encoder.") for key in missing_keys):
    raise RuntimeError(f"Missing required checkpoint keys: {missing_keys}")
if missing_keys:
    print("Loaded public inference checkpoint; sentence encoder weights come from sentence-transformers.")
model.eval()

print("Loaded checkpoint")


## Factual, No-News, And Custom-News Prediction

The factual prediction encodes the observed news events in the historical window. The no-news counterfactual replaces all events with the baseline no-news treatment. The custom-news scenario lets you insert hypothetical event text into the same historical window and ask how the model prediction changes.

In [ ]:
batch = next(iter(test_loader))

seq_x = batch["seq_x"].to(device)
seq_y = batch["seq_y"].to(device)
seq_x_mark = batch["seq_x_mark"].to(device)
news_mask = batch["news_mask"].to(device)

with torch.no_grad():
    factual_pred, _, _, factual_treatment = model(
        seq_x,
        seq_x_mark,
        batch["news_items"],
        news_mask,
        batch["news_metadata"],
    )

    no_news_treatment = model.news_encoder.get_baseline(device).expand(seq_x.size(0), -1)
    no_news_pred, _, _, _ = model(seq_x, seq_x_mark, treatment=no_news_treatment)

history = inverse_transform_predictions(scaler, seq_x[:, :, 0]).squeeze()
truth = inverse_transform_predictions(scaler, seq_y).squeeze()
factual = inverse_transform_predictions(scaler, factual_pred).squeeze()
no_news = inverse_transform_predictions(scaler, no_news_pred).squeeze()
no_news_perturbation = factual - no_news

print(f"Mean factual vs no-news perturbation: {no_news_perturbation.mean():.2f} MW")

In [ ]:
sample_news = extract_sample_news(
    batch["news_items"],
    batch["news_mask"][0],
    batch["news_metadata"],
    batch_idx=0,
)

print(f"News events in historical window: {len(sample_news)}")
for item in sample_news[:5]:
    print(f"- t={item['timestep']:02d} | {item['type']} | score={item['relevance_score']} | {item['text'][:160]}")

## Define A Custom News Scenario

Choose one hypothetical news scenario by changing `selected_scenario_id`. Each scenario is encoded as structured event input and inserted into the same historical window.

In [ ]:
def normalized_time_features(month, day, weekday, hour, minute):
    return np.array([
        month / 12.0 - 0.5,
        day / 31.0 - 0.5,
        weekday / 6.0 - 0.5,
        hour / 23.0 - 0.5,
        minute / 59.0 - 0.5,
    ], dtype=np.float32)


def build_custom_news_inputs(seq_len, batch_size, custom_events):
    news_items = [[[] for _ in range(batch_size)] for _ in range(seq_len)]
    news_metadata = [[[] for _ in range(batch_size)] for _ in range(seq_len)]
    news_mask = torch.zeros(batch_size, seq_len, dtype=torch.bool)

    for event in custom_events:
        timestep = int(event.get("timestep", seq_len - 1))
        timestep = max(0, min(seq_len - 1, timestep))
        batch_idx = int(event.get("batch_idx", 0))
        event_type = event.get("type", "Prediction")
        text = event["text"]
        news_items[timestep][batch_idx].append(f"[{event_type.upper()}] {text}")
        news_metadata[timestep][batch_idx].append({
            "type": event_type,
            "text": text,
            "category": event.get("category", "Custom"),
            "scope": event.get("scope", "State_Wide"),
            "relevance_score": event.get("relevance_score", 10),
            "publication_time": event.get("publication_time", ""),
            "pub_tf": event.get("pub_tf", normalized_time_features(1, 1, 0, 12, 0)),
            "source": event.get("source", "custom scenario"),
            "justification": event.get("justification", "User-defined counterfactual news scenario."),
        })
        news_mask[batch_idx, timestep] = True

    return news_items, news_mask, news_metadata

In [ ]:
CUSTOM_NEWS_SCENARIOS = {
    "storm_outage": {
        "label": "Storm outage and emergency response",
        "events": [{
            "timestep": 47,
            "type": "Prediction",
            "text": "Severe storms are expected to damage parts of the distribution network in New South Wales tomorrow, with outages possible in several suburbs.",
            "category": "Grid_Reliability",
            "scope": "State_Wide",
            "relevance_score": 9,
            "publication_time": "2019-11-15 15:00:00",
            "pub_tf": normalized_time_features(month=11, day=15, weekday=4, hour=15, minute=0),
            "justification": "Grid reliability scenario for counterfactual inference.",
        }],
    },
    "cyclone_landfall": {
        "label": "Tropical cyclone approaching coastline",
        "events": [{
            "timestep": 47,
            "type": "Prediction",
            "text": "A category 3 tropical cyclone is forecast to make landfall on the New South Wales coast tomorrow, with destructive winds and widespread power disruptions expected across the Hunter and Central Coast regions.",
            "category": "Env_Disaster",
            "scope": "State_Wide",
            "relevance_score": 10,
            "publication_time": "2019-02-15 09:00:00",
            "pub_tf": normalized_time_features(month=2, day=15, weekday=4, hour=9, minute=0),
            "justification": "Natural disaster scenario for counterfactual inference.",
        }],
    },
    "bushfire_evacuation": {
        "label": "Bushfire evacuation and power shutdown",
        "events": [{
            "timestep": 47,
            "type": "Fact",
            "text": "Catastrophic bushfire conditions have forced mass evacuations across multiple New South Wales regions, with power lines de-energized and emergency shelters activated.",
            "category": "Env_Disaster",
            "scope": "State_Wide",
            "relevance_score": 10,
            "publication_time": "2019-12-21 10:00:00",
            "pub_tf": normalized_time_features(month=12, day=21, weekday=5, hour=10, minute=0),
            "justification": "Bushfire disaster scenario for counterfactual inference.",
        }],
    },
    "stay_at_home_restrictions": {
        "label": "Stay-at-home restrictions announced",
        "events": [{
            "timestep": 47,
            "type": "Fact",
            "text": "New South Wales will enter stay-at-home restrictions tomorrow, with many offices and non-essential venues closed and residents spending more time at home.",
            "category": "Soc_Calendar",
            "scope": "State_Wide",
            "relevance_score": 10,
            "publication_time": "2019-07-10 18:00:00",
            "pub_tf": normalized_time_features(month=7, day=10, weekday=2, hour=18, minute=0),
            "justification": "Mobility restriction scenario for counterfactual inference.",
        }],
    },
    "transport_strike": {
        "label": "Public transport workers strike",
        "events": [{
            "timestep": 47,
            "type": "Fact",
            "text": "All Sydney train and bus workers will go on a 24-hour strike tomorrow, forcing many businesses to allow employees to work from home and reducing commercial district activity.",
            "category": "Soc_Economic",
            "scope": "State_Wide",
            "relevance_score": 8,
            "publication_time": "2019-05-06 18:00:00",
            "pub_tf": normalized_time_features(month=5, day=6, weekday=0, hour=18, minute=0),
            "justification": "Economic disruption scenario for counterfactual inference.",
        }],
    },
    "industrial_closure": {
        "label": "Major industrial plant closure",
        "events": [{
            "timestep": 47,
            "type": "Fact",
            "text": "The Tomago aluminium smelter in New South Wales has shut down all production lines due to a contract dispute, removing one of the state's largest single electricity loads from the grid.",
            "category": "Soc_Economic",
            "scope": "State_Wide",
            "relevance_score": 9,
            "publication_time": "2019-08-12 11:00:00",
            "pub_tf": normalized_time_features(month=8, day=12, weekday=0, hour=11, minute=0),
            "justification": "Industrial demand reduction scenario for counterfactual inference.",
        }],
    },
}

selected_scenario_id = "storm_outage"
selected_scenario = CUSTOM_NEWS_SCENARIOS[selected_scenario_id]
custom_events = selected_scenario["events"]

custom_news_items, custom_news_mask, custom_news_metadata = build_custom_news_inputs(
    seq_len=args.seq_len,
    batch_size=seq_x.size(0),
    custom_events=custom_events,
)

with torch.no_grad():
    custom_pred, _, _, custom_treatment = model(
        seq_x,
        seq_x_mark,
        custom_news_items,
        custom_news_mask.to(device),
        custom_news_metadata,
    )

custom = inverse_transform_predictions(scaler, custom_pred).squeeze()
custom_vs_no_news = custom - no_news
custom_vs_factual = custom - factual

print(f"Selected scenario: {selected_scenario['label']}")
print(f"Mean custom vs no-news perturbation: {custom_vs_no_news.mean():.2f} MW")
print(f"Mean custom vs factual difference: {custom_vs_factual.mean():.2f} MW")

## Visualize The Counterfactual Difference

In [ ]:
hist_x = np.arange(len(history))
pred_x = np.arange(len(history), len(history) + len(truth))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(hist_x, history, color="0.35", label="Historical load")
ax1.plot(pred_x, truth, color="black", label="True future load")
ax1.plot(pred_x, factual, color="#1f77b4", label="Factual forecast")
ax1.plot(pred_x, no_news, color="#2ca02c", label="No-news counterfactual")
ax1.plot(pred_x, custom, color="#9467bd", label="Custom-news scenario")
ax1.axvline(len(history) - 1, color="0.6", linestyle="--", linewidth=1)
ax1.set_xlabel("Time step")
ax1.set_ylabel("Load (MW)")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()
ax2.bar(pred_x, custom_vs_no_news, color="#d62728", alpha=0.22, label="Custom - no-news")
ax2.axhline(0, color="0.4", linewidth=1)
ax2.set_ylabel("Estimated perturbation (MW)")

fig.tight_layout()
plt.show()